# AssistAnt: Multi-Agent AI Team

**Angeles University Foundation · PTF60 Project · BS Computer Science 4**

This notebook builds a small team of AI agents that reads local technical documents and writes a fact-checked summary report as a PDF. It is based on the AssistAnt app in this repository.

## Motivation

I already run my daily life with a personal assistant built on Claude. It keeps my tasks, tracks my expenses and plans my days, and it works well because it is tailored to me and my routines.

AssistAnt asks the next question: could an assistant like that become a product other people use? It is a public demo of the idea. It shows the parts that would have to work for everyone, not just for me: a team of agents that splits up the work, answers grounded in the user's own notes, and automation that asks before making risky changes. This notebook rebuilds AssistAnt's backend so the design can be studied, tested and extended.

## What AssistAnt can help with

- **Wrap up the day:** summarizes what got done, closes finished reminders and carries leftovers to tomorrow.
- **Morning brief:** what matters today and what is due in the next few days.
- **Deadlines:** lists what is due this week and finds deadlines hidden in notes.
- **Tasks:** adds a task to any day of the week.
- **Groceries:** adds items to the list and skips duplicates.
- **Chores:** keeps repeating chores, like a weekly clean, on schedule.
- **Money:** logs investment accounts and shows the change since last time.
- **Questions about your notes:** answers from your own notes, with numbered sources.

## How this notebook works

**The team**
- **Orchestrator** (the manager): reads the request, decides how hard it is and assigns the work.
- **Reader**: opens the documents through an MCP server.
- **Notes Scout**: finds the evidence with RAG search.
- **Writer**: drafts the report from the evidence only.
- **Checker**: fact-checks the draft and sends it back if something is wrong.
- **Publisher**: builds the final report and the PDF.

**Where each requirement is covered**

| Requirement | Part |
|---|---|
| Manager routing | 4c, 5b |
| MCP file tools | 3b |
| RAG fact-checking | 3c, 4e, 4f |
| Short-term and long-term memory | 5a |
| Agents reviewing each other's work | 4e, 4f |
| PDF report | 7 |

No API key? The notebook still runs. It falls back to free models that need no key, then to simple rules.

## Part 1: Environment Setup

Install the libraries, then load the Gemini API key from Colab Secrets.

In [1]:
# Part 1 · Every install, in one cell. Run this first.
# The agent framework is LangGraph only. CrewAI isn't installed: it isn't used here, and its pinned versions clash with recent LangGraph and ChromaDB.
!pip install -q langgraph google-genai sentence-transformers chromadb mcp reportlab requests

In [2]:
# Gemini API key from Colab Secrets: click the key icon in the sidebar, add GEMINI_API_KEY and allow this notebook.
# Never paste a key into a cell. Outside Colab, the GEMINI_API_KEY environment variable is used.
import os

GEMINI_API_KEY = None
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:  # not on Colab, or the secret isn't shared with this notebook
    pass
GEMINI_API_KEY = GEMINI_API_KEY or os.environ.get('GEMINI_API_KEY')
print('Gemini key:', 'found' if GEMINI_API_KEY else 'not set, so the router will use the backups')

# Optional backup: a FreeLLMAPI router (github.com/tashfeenahmed/freellmapi), one endpoint over many free providers.
# Add FREELLMAPI_URL (for example http://your-host:3001) and FREELLMAPI_KEY (its unified key) the same way.
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

FREELLMAPI_URL, FREELLMAPI_KEY = secret('FREELLMAPI_URL'), secret('FREELLMAPI_KEY')
print('FreeLLMAPI router:', 'found' if FREELLMAPI_URL and FREELLMAPI_KEY else 'not set (optional)')

Gemini key: not set, so the router will use the backups
FreeLLMAPI router: not set (optional)


## Part 2: Architecture Overview

The Orchestrator reads the request, picks a skill and rates the task as easy, medium or hard. The rating decides which model tier to use, how many documents to read and how many review rounds to allow. Easy tasks skip the Reader.

The agents share notes in the graph state (short-term memory). The Checker can send the draft back to the Writer. Standing rules and past reports are kept in long-term memory.

```
                   medium / hard
User -> Orchestrator ----------> Reader ---+
            |                              v
            +-------- easy ----------> Notes Scout -> Writer -> Checker -> Publisher -> PDF
                                                        ^         |
                                                        +---------+
                                                   sent back if there are issues

Reader       reads files through the MCP server (list, search, read)
Notes Scout  finds evidence with RAG (embeddings + ChromaDB)
Memory       MemorySaver (this conversation), InMemoryStore (rules and past reports)
```

## Part 3: MCP and RAG Configuration

**3a. Documents and skills.** The sample documents describe the AssistAnt app. You can add your own `.md` or `.txt` files to the `docs/` folder. Existing files are never overwritten.

In [3]:
# Part 3a · Workspace: local technical documents + skills.
import re, json, sys, time
from pathlib import Path

WORK = Path('/content/assistant') if Path('/content').exists() else Path.cwd() / 'assistant_work'
DOCS, SKILLS, OUT = WORK / 'docs', WORK / 'skills', WORK / 'outputs'
for d in (DOCS, SKILLS, OUT):
    d.mkdir(parents=True, exist_ok=True)
rel = lambda p: Path(p).relative_to(WORK.parent).as_posix()  # short paths in printouts

SAMPLE_DOCS = {
'architecture.md': """# AssistAnt architecture

AssistAnt is a personal agent that wraps up the day in Notion and plans tomorrow. It runs as a Next.js website and an installable app from one project.

## Orchestrator
Every request goes to the Orchestrator. It matches the request against routines, which are markdown files with trigger phrases. If a routine matches, its flow runs. If none matches, the Orchestrator plans the request itself with the unplanned flow and offers to save a new routine.

## The crew
The Orchestrator hands work to eleven subagents. Each agent has a tier that decides which line of models it asks for. Agents on the code tier never call a model, so they are instant and free.

- Reader (code tier) reads the Tasks page, Reminders, Notes and the Stocks sheet.
- Summarizer (smart tier) writes a short, kind summary of the day.
- Matcher (code tier) matches finished items to the right Reminders.
- Notes Scout (best tier) finds deadlines in notes that never made it into Reminders.
- Chore Keeper (code tier) keeps chore cadences going.
- Planner (code tier) builds tomorrow from deadlines due within three days, leftovers and the routine.
- Writer (code tier) is the only agent allowed to change Notion or Sheets.
- Checker (code tier) re-reads everything after the Writer finishes and sends back anything that looks wrong.
- Grocery Clerk (fast tier) adds grocery items and skips duplicates.
- Briefer (smart tier) writes the morning brief.
- Portfolio Clerk (code tier) logs two brokerage accounts and a crypto wallet, converting the wallet to dollars.

## Runs and events
Each request is a run. A run emits events such as the plan, messages between agents, work done and the result. The browser streams these events live with Server-Sent Events. Events are kept so a page opened later can replay them.

## Approvals
Risky changes wait for the user. The Writer holds new guessed Reminders and the deletion of today's block until the user approves them. Scheduled runs keep their approvals waiting until the app is opened.
""",
'model-router.md': """# Model router

Each agent asks the router for a tier: fast, smart or best. The router walks that tier's chain of providers and uses the first one that is available.

## Chains
- The best chain is Claude Code on the user's Pro or Max plan, Groq gpt-oss-120b, the free providers from the awesome-free-llm-apis list, OpenRouter, then a local Ollama model.
- The smart chain is Gemini Flash, Groq gpt-oss-120b, the free list, OpenRouter, Ollama, then the paid Claude API, which is off by default.
- The fast chain is Groq gpt-oss-20b, Gemini Flash-Lite, Groq Qwen, the free list, OpenRouter, Ollama, then the Claude API.

## Health checks
Before calling a provider the router checks its health. A provider can be ok, full when used up today, off when switched off in Sources, nokey when no API key is set, or offline when Ollama or Claude Code is not running. Only providers that are ok are called.

## Limits and usage
Usage is counted per provider per day and resets at midnight in the configured time zone. Providers with a daily cap, such as Gemini Flash at 20 requests a day or Groq at 1,000, are skipped once the cap is reached. A 429 response marks the provider exhausted for the rest of the day.

## Fallback to plain rules
If every provider in the chain fails or is unavailable, the router returns nothing and the agent falls back to plain rules. Every flow is written so it can finish without a model, which keeps AssistAnt working offline and for free.

## Keyless gateways
Three gateways from the free list, Kilo Code, LLM7.io and OVHcloud AI Endpoints, work without an API key. They are off by default because the list says they may log prompts. The public demo switches them on because it only handles fake data.
""",
'routines.md': """# Routines

Routines are markdown files in the routines folder. Each one starts with a small front matter block with a name, a flow and a list of triggers.

Triggers are case-insensitive regular expressions. The Orchestrator tests every routine's triggers against the request and runs the first routine that matches. The body of the file describes the steps in plain language.

## Built-in routines
- Finish day is the nightly wrap-up: summarize the day, mark finished Reminders done, find deadlines hidden in notes, keep chores going, build tomorrow, then check the result.
- Morning brief covers what matters today, what is due within three days, and anything in the Worries note.
- What's due lists open Reminders due in the next seven days, sorted by date then priority.
- Groceries adds items to the Grocery List and skips duplicates, including obvious synonyms.
- Add a task puts a new task in that day's new-tasks area, never inside the routine.
- Portfolio asks for the day's figures for two brokerage accounts and a crypto wallet and adds one row to the Daily Snapshot.

## Learning new routines
When no routine matches, the Orchestrator answers the request and drafts a reusable routine. It asks for approval before saving it, so a new routine never appears without the user's OK.

## Schedule
Finish day runs at 22:10 and the morning brief at 07:00, in the configured time zone. The scheduler checks the clock every 20 seconds and runs each job once per day.
""",
'knowledge-base.md': """# Knowledge Base and chat

The Knowledge Base holds notes the user uploads, pastes or copies from Notion. Each note is stored as a JSON file.

## Chunking
Notes are split on blank lines and headings, then packed into chunks of about 900 characters. Paragraphs longer than 1,400 characters are cut into 900-character pieces.

## Search
Search is local keyword ranking with BM25, using k1 = 1.4 and b = 0.75. It is free, private and works offline. Common stop words are ignored. Re-uploading a note with the same title replaces the old copy.

## Chat
Chat messages are routed in two ways. Questions go to the Knowledge Base, where the top passages are retrieved and the answer cites them as numbered sources. Requests to do something, such as add or plan, go to the crew as a run. If the crew is already busy, the chat says so instead of starting a second run.

## Limits
Uploads accept text formats only, which are Markdown, text, CSV, JSON and HTML, up to 2 MB per file. HTML is stripped of scripts, styles and tags before indexing.
""",
'public-demo.md': """# Public demo mode

Setting PUBLIC_DEMO=1 turns AssistAnt into a public demo. It is set only on the public host, while the owner's own copy runs locally with real Notion and keys.

## What changes
- The workspace is always the fake sample workspace, whatever the environment holds. The Connectors panel shows Notion and Google Sheets as connected to sample data.
- Only the keyless model gateways are offered, switched on, and no API key is ever sent. When they are busy, flows fall back to plain rules.
- The scheduler is off, and runs and chat are rate limited per visitor and per IP address.

## Visitor isolation
Each browser gets a random visitor ID in a cookie, set by middleware before the first API call. Chats, the Knowledge Base, model usage, the fake workspace, runs and learned routines are kept per visitor, so visitors never see or block each other.

## Hosting
The demo runs on the free tier of Render. The service sleeps after about 15 minutes without traffic, the first request afterwards takes 30 to 60 seconds, and the disk is wiped on restart, so visitor data resets. These limits are fine for a demo but not for daily personal use.
""",
'workspace.md': """# Workspaces: Notion, Sheets and the demo

All agents read and change data through one Workspace interface, so flows never call Notion directly. The interface covers the day blocks on the Tasks page, Reminders, Notes, the Grocery List and the portfolio snapshot.

## Notion
The Notion workspace needs NOTION_TOKEN plus the Tasks page, Reminders database and Notes page IDs. It retries on 429 responses with a growing delay.

## Google Sheets
The Stocks sheet is reached with a Google service account. Each Daily Snapshot row holds the date, the two brokerage accounts and the wallet in USD, the exchange rate and notes.

## Demo workspace
The demo workspace is a fictional user's Tasks page with sample tasks, reminders and notes. It lives in memory and is seeded once per day.

## Dates
All dates are in YYYY-MM-DD form, in the configured time zone (UTC by default). A deadline is labelled due today, or with a short date such as due Thu, Oct 29.
""",
}

# Skills work like AssistAnt's routines: front matter with trigger regexes, then the steps.
SAMPLE_SKILLS = {
'quick-answer.md': """---
name: Quick answer
triggers: ^(what|which|who|when|where|how|is|does|do|can)\\b.*\\?$
sections: Answer, Supporting facts
---
Answer the question directly in a short paragraph, then list the supporting facts as bullets. Every factual sentence cites its source as [S#].
""",
'tech-report.md': """---
name: Technical report
triggers: report, summar, overview, write.?up, document, explain
sections: Summary, Key findings, How it works
---
Write a technical report for an engineer new to the system. Start with a short summary, then the key findings as bullets, then how it works step by step. Every factual sentence cites its source as [S#].
""",
}

for folder, files in ((DOCS, SAMPLE_DOCS), (SKILLS, SAMPLE_SKILLS)):
    for name, text in files.items():
        if not (folder / name).exists():  # idempotent: never overwrite your edits
            (folder / name).write_text(text, encoding='utf-8')

print('Documents:', sorted(p.name for p in DOCS.iterdir()))
print('Skills:   ', sorted(p.name for p in SKILLS.iterdir()))

Documents: ['architecture.md', 'knowledge-base.md', 'model-router.md', 'public-demo.md', 'routines.md', 'workspace.md']
Skills:    ['quick-answer.md', 'tech-report.md']


**3b. MCP server.** A small server built with the official Python MCP SDK. It offers three tools, `list_files`, `search_files` and `read_file`, and it can only read files inside `docs/`. The agents use it for all file access. The test cell reads a file through it before any agent runs.

In [4]:
# Part 3b · MCP server: the only way agents reach local files.
SERVER = WORK / 'mcp_docs_server.py'
SERVER.write_text(r"""
import json, os, re
from pathlib import Path
try:  # MCP SDK 2.x renamed FastMCP to MCPServer
    from mcp.server.mcpserver import MCPServer as FastMCP
except ImportError:  # 1.x
    from mcp.server.fastmcp import FastMCP

ROOT = Path(os.environ.get('DOCS_ROOT', 'docs')).resolve()
TEXT = {'.md', '.txt', '.rst', '.csv', '.json'}
mcp = FastMCP('assistant-docs')

def _safe(path: str) -> Path:
    full = (ROOT / path).resolve()
    if not full.is_relative_to(ROOT) or not full.is_file():  # no ../ tricks, files only
        raise ValueError(f'No such document: {path}')
    return full

def _docs():
    return sorted(p for p in ROOT.rglob('*') if p.is_file() and p.suffix.lower() in TEXT)

@mcp.tool()
def list_files() -> str:
    'List the local technical documents.'
    return json.dumps([{'path': p.relative_to(ROOT).as_posix(), 'bytes': p.stat().st_size} for p in _docs()])

@mcp.tool()
def search_files(query: str, limit: int = 5) -> str:
    'Rank documents by how often the query words appear (file names count triple).'
    words = [w for w in re.findall(r'[a-z0-9]+', query.lower()) if len(w) > 2]
    scored = []
    for p in _docs():
        name, text = p.stem.lower(), p.read_text(encoding='utf-8', errors='ignore').lower()
        score = sum(3 * name.count(w) + text.count(w) for w in words)
        if score:
            scored.append({'path': p.relative_to(ROOT).as_posix(), 'score': score})
    return json.dumps(sorted(scored, key=lambda x: -x['score'])[:max(1, limit)])

@mcp.tool()
def read_file(path: str, max_chars: int = 20000) -> str:
    'Read one document (UTF-8 text).'
    text = _safe(path).read_text(encoding='utf-8', errors='ignore')
    return json.dumps({'path': path, 'chars': len(text), 'text': text[:max_chars]})

if __name__ == '__main__':
    mcp.run()  # stdio transport
""", encoding='utf-8')
print('MCP server written to', rel(SERVER))

MCP server written to assistant_work/mcp_docs_server.py


In [5]:
# Part 3b · MCP client: one long-lived stdio session shared by every agent.
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

class DocsMCP:
    async def start(self):
        params = StdioServerParameters(command=sys.executable, args=[str(SERVER)], env={**os.environ, 'DOCS_ROOT': str(DOCS)})
        self.stack = AsyncExitStack()
        errlog = self.stack.enter_context(open(WORK / 'mcp_server.log', 'w'))  # Jupyter's stderr can't be handed to a subprocess
        read, write = await self.stack.enter_async_context(stdio_client(params, errlog=errlog))
        self.session = await self.stack.enter_async_context(ClientSession(read, write))
        await self.session.initialize()
        return [t.name for t in (await self.session.list_tools()).tools]

    async def call(self, tool, **args):
        result = await self.session.call_tool(tool, args)
        text = result.content[0].text if result.content else ''
        if getattr(result, 'is_error', None) or getattr(result, 'isError', False):
            raise RuntimeError(f'MCP {tool} failed: {text}')
        return json.loads(text)

DOCS_MCP = DocsMCP()
print('MCP tools available:', await DOCS_MCP.start())

MCP tools available: ['list_files', 'search_files', 'read_file']


In [6]:
# Part 3b · Quick test: read a local file through MCP before any agent is involved.
files = await DOCS_MCP.call('list_files')
print('list_files →', [f['path'] for f in files])
doc = await DOCS_MCP.call('read_file', path='architecture.md')
print(f"read_file('architecture.md') → {doc['chars']} characters. First lines:")
print('   ' + '\n   '.join(doc['text'].splitlines()[:3]))
print('search_files("model fallback") →', await DOCS_MCP.call('search_files', query='model fallback', limit=3))

list_files →

 ['architecture.md', 'knowledge-base.md', 'model-router.md', 'public-demo.md', 'routines.md', 'workspace.md']
read_file('architecture.md') → 2030 characters. First lines:
   # AssistAnt architecture
   
   AssistAnt is a personal agent that wraps up the day in Notion and plans tomorrow. It runs as a Next.js website and an installable app from one project.
search_files("model fallback") → [{'path': 'model-router.md', 'score': 7}, {'path': 'architecture.md', 'score': 2}, {'path': 'public-demo.md', 'score': 2}]


**3c. RAG.** The documents are split into chunks of about 900 characters, turned into embeddings with `all-MiniLM-L6-v2` and stored in ChromaDB. Running this cell again does not create duplicates.

In [7]:
# Part 3c · RAG: local embeddings + ChromaDB.
os.environ['ANONYMIZED_TELEMETRY'] = 'False'
import chromadb
from sentence_transformers import SentenceTransformer

EMBED = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
CHROMA = chromadb.PersistentClient(path=str(WORK / 'chroma'))
KB = CHROMA.get_or_create_collection('assistant_docs', metadata={'hnsw:space': 'cosine'})

def chunk(text):
    """Port of AssistAnt's lib/kb.ts chunk(): split on blank lines and headings, pack to ~900 chars."""
    paras = [p.strip() for p in re.split(r'\n\s*\n|\n(?=#)', text.replace('\r', '')) if p.strip()]
    out, buf = [], ''
    for p in paras:
        if buf and len(buf) + len(p) > 900:
            out.append(buf); buf = ''
        if len(p) > 1400:
            out += [p[k:k + 900] for k in range(0, len(p), 900)]; continue
        buf = buf + '\n\n' + p if buf else p
    return out + ([buf] if buf else [])

def embed(texts):
    return EMBED.encode(list(texts), normalize_embeddings=True)

async def index_docs():
    for f in await DOCS_MCP.call('list_files'):
        parts = chunk((await DOCS_MCP.call('read_file', path=f['path']))['text'])
        KB.delete(where={'source': f['path']})  # drop stale chunks if a file got shorter
        KB.upsert(ids=[f"{f['path']}::{i}" for i in range(len(parts))], documents=parts,
                  embeddings=embed(parts).tolist(), metadatas=[{'source': f['path'], 'chunk': i} for i in range(len(parts))])
    return KB.count()

def retrieve(query, k=5):
    r = KB.query(query_embeddings=embed([query]).tolist(), n_results=min(k, KB.count()), include=['documents', 'metadatas', 'distances'])
    return [{'id': i, 'source': m['source'], 'chunk': m['chunk'], 'text': d, 'score': round(1 - dist, 3)}
            for i, d, m, dist in zip(r['ids'][0], r['documents'][0], r['metadatas'][0], r['distances'][0])]

print('Indexed chunks:', await index_docs())
for h in retrieve('What happens when every model fails?', 3):
    print(f"  {h['score']:.2f}  {h['source']}#{h['chunk']}  {h['text'][:80].replace(chr(10), ' ')}…")

Indexed chunks: 14
  0.27  architecture.md#0  # AssistAnt architecture  AssistAnt is a personal agent that wraps up the day in…
  0.21  model-router.md#1  ## Limits and usage Usage is counted per provider per day and resets at midnight…
  0.19  public-demo.md#1  ## Hosting The demo runs on the free tier of Render. The service sleeps after ab…


## Part 4: Building the AI Team

**4a. Model router.** Each agent asks for a tier: fast, smart or best. **Google Gemini is the default** for every tier. If Gemini has no key, hits its limit or fails, the backups take over in order: a [FreeLLMAPI](https://github.com/tashfeenahmed/freellmapi) router (one endpoint over many free providers, if you set `FREELLMAPI_URL` and `FREELLMAPI_KEY`), then free models that need no key from the [awesome-free-llm-apis](https://github.com/mnfst/awesome-free-llm-apis) list, then simple rules. A model that hits its limit is skipped for the rest of the session.

In [8]:
# Part 4a · Model router with fallbacks.
import requests
from google import genai
from google.genai import types as gtypes

# 90 s per request and one retry, so a slow or stuck Gemini call falls through to the backups.
GEMINI = genai.Client(api_key=GEMINI_API_KEY, http_options=gtypes.HttpOptions(
    timeout=90_000, retry_options=gtypes.HttpRetryOptions(attempts=2))) if GEMINI_API_KEY else None

def _newest(pattern, env, default):
    """Use the env override, else the newest matching model your key can see, else the app's default."""
    if os.environ.get(env):
        return os.environ[env]
    try:
        names = [m.name.split('/')[-1] for m in GEMINI.models.list()]
        found = sorted((n for n in names if re.fullmatch(pattern, n)), key=lambda n: [int(x) for x in re.findall(r'\d+', n)])
        return found[-1] if found else default
    except Exception:
        return default

GEMINI_LITE = _newest(r'gemini-\d+(\.\d+)?-flash-lite', 'GEMINI_LITE_MODEL', 'gemini-3.5-flash-lite') if GEMINI else None
GEMINI_FLASH = _newest(r'gemini-\d+(\.\d+)?-flash', 'GEMINI_FLASH_MODEL', 'gemini-3.8-flash') if GEMINI else None

class RateLimited(Exception):
    pass

def _gemini(model):
    def call(prompt, system):
        def go(think):
            extra = {} if think else {'thinking_config': gtypes.ThinkingConfig(thinking_budget=0)}
            return GEMINI.models.generate_content(model=model, contents=prompt,
                                                  config=gtypes.GenerateContentConfig(system_instruction=system, temperature=0.2, **extra))
        try:
            try:
                r = go(think=False)  # skip the long "thinking" pass: much faster for short reports
            except Exception as e:
                if 'thinking' not in str(e).lower():
                    raise
                r = go(think=True)   # this model can't turn thinking off
        except Exception as e:
            if '429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e):
                raise RateLimited() from e
            raise
        return (r.text or '').strip()
    return call

def _openai_compatible(url, model, key=None):
    def call(prompt, system):
        messages = ([{'role': 'system', 'content': system}] if system else []) + [{'role': 'user', 'content': prompt}]
        headers = {'Authorization': f'Bearer {key}'} if key else {}
        r = requests.post(url, json={'model': model, 'messages': messages, 'temperature': 0.2}, headers=headers, timeout=90)
        if r.status_code == 429:
            raise RateLimited()
        r.raise_for_status()
        return (r.json()['choices'][0]['message']['content'] or '').strip()
    return call

# Default: Google Gemini (both models, every tier). Backups: keyless gateways from the same free list the app uses.
# Keyless gateways come from the same free list the app uses. They may log prompts, which is fine for these docs.
PROVIDERS = {
    'gemini-lite':  {'name': f'Gemini {GEMINI_LITE}' if GEMINI else 'Gemini Flash-Lite',  'ready': bool(GEMINI), 'call': _gemini(GEMINI_LITE) if GEMINI else None},
    'gemini-flash': {'name': f'Gemini {GEMINI_FLASH}' if GEMINI else 'Gemini Flash', 'ready': bool(GEMINI), 'call': _gemini(GEMINI_FLASH) if GEMINI else None},
    # FreeLLMAPI routes each request to the best healthy model among the providers you added to it ('auto').
    'freellmapi': {'name': 'FreeLLMAPI router', 'ready': bool(FREELLMAPI_URL and FREELLMAPI_KEY),
                   'call': _openai_compatible((FREELLMAPI_URL or '').rstrip('/').removesuffix('/v1') + '/v1/chat/completions', 'auto', FREELLMAPI_KEY)},
    'kilo': {'name': 'Kilo (keyless)', 'ready': True, 'call': _openai_compatible('https://api.kilo.ai/api/gateway/chat/completions', 'kilo-auto/free')},
    'llm7': {'name': 'LLM7 (keyless)', 'ready': True, 'call': _openai_compatible('https://api.llm7.io/v1/chat/completions', 'gpt-oss:20b')},
    'ovh':  {'name': 'OVH (keyless)',  'ready': True, 'call': _openai_compatible('https://oai.endpoints.kepler.ai.cloud.ovh.net/v1/chat/completions', 'gpt-oss-120b')},
}
CHAINS = {
    'fast':  ['gemini-lite', 'gemini-flash', 'freellmapi', 'kilo', 'llm7', 'ovh'],
    'smart': ['gemini-flash', 'gemini-lite', 'freellmapi', 'kilo', 'llm7', 'ovh'],
    'best':  ['gemini-flash', 'gemini-lite', 'freellmapi', 'kilo', 'ovh', 'llm7'],
}
USAGE, EXHAUSTED, FAILS = {}, set(), {}

def ask(tier, prompt, system=None):
    """Return (text, model name). (None, 'plain rules') when every provider in the chain is unavailable."""
    problems = []
    for pid in CHAINS[tier]:
        p = PROVIDERS[pid]
        if not p['ready'] or pid in EXHAUSTED or FAILS.get(pid, 0) >= 2:
            continue
        try:
            text = p['call'](prompt, system)
            if not text:
                raise ValueError('empty reply')
            USAGE[pid] = USAGE.get(pid, 0) + 1
            if problems:
                print(f"   Router: {problems[0]}, so this went to {p['name']}.")
            return text, p['name']
        except RateLimited:
            EXHAUSTED.add(pid); problems.append(f"{p['name']} hit its limit")
        except Exception as e:
            FAILS[pid] = FAILS.get(pid, 0) + 1; problems.append(f"{p['name']} failed ({type(e).__name__})")
    print('   Router: no model is available right now, so this step used plain rules.')
    return None, 'plain rules'

def parse_json(text, fallback):
    """Port of router.ts parseJSON: pull the first JSON value out of a model reply."""
    m = re.search(r'```(?:json)?\s*([\s\S]*?)```', text or '')
    raw = m.group(1) if m else (text or '')
    start = min([i for i in (raw.find('{'), raw.find('[')) if i >= 0], default=-1)
    if start < 0:
        return fallback
    try:
        return json.loads(raw[start:max(raw.rfind('}'), raw.rfind(']')) + 1])
    except Exception:
        return fallback

print('Default: Google Gemini' + ('' if GEMINI else ' (no key found, so the backups answer)') + '. Backups: FreeLLMAPI router'
      + ('' if PROVIDERS['freellmapi']['ready'] else ' (not set)') + ', then Kilo, LLM7, OVH, then plain rules.')
for tier, chain in CHAINS.items():
    print(f"{tier:>5}: " + ' → '.join(PROVIDERS[p]['name'] + ('' if PROVIDERS[p]['ready'] else ' (no key)') for p in chain) + ' → plain rules')

Default: Google Gemini (no key found, so the backups answer). Backups: FreeLLMAPI router (not set), then Kilo, LLM7, OVH, then plain rules.
 fast: Gemini Flash-Lite (no key) → Gemini Flash (no key) → FreeLLMAPI router (no key) → Kilo (keyless) → LLM7 (keyless) → OVH (keyless) → plain rules
smart: Gemini Flash (no key) → Gemini Flash-Lite (no key) → FreeLLMAPI router (no key) → Kilo (keyless) → LLM7 (keyless) → OVH (keyless) → plain rules
 best: Gemini Flash (no key) → Gemini Flash-Lite (no key) → FreeLLMAPI router (no key) → Kilo (keyless) → OVH (keyless) → LLM7 (keyless) → plain rules


**4b. Shared state.** All agents read and write one state object. `notes` is the shared scratchpad and `trace` records every hand-off.

In [9]:
# Part 4b · Shared team state + hand-off log.
import operator
from typing import Annotated, TypedDict

class TeamState(TypedDict, total=False):
    request: str
    user_id: str
    run_id: str
    skill: dict
    difficulty: str
    tier: str
    plan: dict
    rules: list
    history: list
    files: list
    verified: list
    evidence: list
    draft: str
    review: dict
    round: int
    report: str
    models: Annotated[list, operator.add]  # every model that answered, across the task
    notes: Annotated[list, operator.add]   # short-term scratchpad the agents share
    trace: Annotated[list, operator.add]   # every hand-off, for the PDF appendix

ICON = {'you': 'You', 'orch': 'Orchestrator', 'reader': 'Reader', 'scout': 'Notes Scout', 'writer': 'Writer',
        'checker': 'Checker', 'publisher': 'Publisher', 'memory': 'Memory'}

def handoff(frm, to, label, text=''):
    print(f"{ICON[frm]} → {ICON[to]} · {label}" + (f"\n      {text}" if text else ''))
    return {'from': frm, 'to': to, 'label': label, 'text': text}

def work(agent, text, model='plain code'):
    print(f"   [{ICON[agent]} · {model}] {text}")

def note(state, agent, text, **extra):
    return {'agent': agent, 'text': text, 'run': state['run_id'], **extra}

def join_citations(text):
    """Move citations a model put on their own line back onto the end of the previous line."""
    return re.sub(r'[ \t]*\n[ \t]*((?:\[[S\d,\s]+\][ \t]*)+)(?=\n|$)', r' \1', text)

def sentences(text):
    """Split docs or drafts into sentences, dropping headings, table rows and list markers."""
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith(('#', '|', '---', '```')):
            continue
        line = re.sub(r'^([-*]|\d+\.)\s+', '', line)
        out += [s.strip() for s in re.split(r'(?<=[.!?])\s+(?=[A-Z0-9])', line) if s.strip()]
    return out

**4c. Orchestrator.** Saves new standing rules, recalls memory, picks a skill, rates the difficulty and assigns the agents.

In [10]:
# Part 4c · Orchestrator: skill + difficulty → tier and lineup.
ORCH_PROMPT = """You are the Orchestrator of a team that writes grounded technical reports from local documents.
Rate how hard the request is. easy = one fact from one document. medium = one topic across a few documents.
hard = a broad report across many documents, with comparisons, chains or risks.
Reply with only JSON: {"difficulty": "easy" | "medium" | "hard", "why": "<one short sentence>"}"""

# What each difficulty gets: model tier, files Reader opens, passages Scout retrieves, review rounds.
PLAN = {
    'easy':   {'tier': 'fast',  'read': 0, 'k': 5,  'rounds': 1},
    'medium': {'tier': 'smart', 'read': 3, 'k': 7,  'rounds': 2},
    'hard':   {'tier': 'best',  'read': 5, 'k': 10, 'rounds': 2},
}
RULE_START = re.compile(r'^(from now on|always|never|remember to|please always)\b', re.I)

def load_skills():
    """Same front-matter format as AssistAnt's routines/*.md."""
    skills = []
    for f in sorted(SKILLS.glob('*.md')):
        m = re.match(r'^---\n([\s\S]*?)\n---\n?([\s\S]*)$', f.read_text(encoding='utf-8'))
        meta = dict(l.split(':', 1) for l in (m.group(1) if m else '').splitlines() if ':' in l)
        meta = {k.strip(): v.strip() for k, v in meta.items()}
        skills.append({'slug': f.stem, 'name': meta.get('name', f.stem), 'steps': (m.group(2) if m else '').strip(),
                       'triggers': [re.compile(t.strip(), re.I) for t in meta.get('triggers', '').split(',') if t.strip()],
                       'sections': [s.strip() for s in meta.get('sections', 'Summary').split(',')]})
    return skills

def match_skill(task):
    skills = load_skills()
    return next((s for s in skills if any(t.search(task) for t in s['triggers'])), next(s for s in skills if s['slug'] == 'tech-report'))

def rule_difficulty(task, hits):
    score = (len(task.split()) > 18) \
          + bool(re.search(r'\b(compare|architecture|end.to.end|every|detailed|in.depth|trade.?offs?|risks?|fallbacks?|chains?)\b', task, re.I)) \
          + (len({h['source'] for h in hits if h['score'] > 0.3}) >= 3)
    return ['easy', 'medium', 'hard'][min(score, 2)]

async def orchestrator(state: TeamState):
    uid = state['user_id']
    trace = [handoff('you', 'orch', 'New request', state['request'])]

    # Long-term memory: save standing rules, then recall rules and past reports.
    parts = re.split(r'(?<=[.!?])\s+', state['request'].strip())
    new_rules = [p for p in parts if RULE_START.match(p)]
    task = ' '.join(p for p in parts if p not in new_rules) or state['request']
    for r in new_rules:
        remember_rule(uid, r)
    if new_rules:
        trace.append(handoff('orch', 'memory', 'Save standing rule', ' · '.join(new_rules)))
    rules, history = recall_rules(uid), recall_history(uid)
    if rules or history:
        last = f" Last report: “{history[0]['request'][:60]}” on {history[0]['at'][:10]}." if history else ''
        work('orch', f"Long-term memory for {uid}: {len(rules)} rule(s), {len(history)} past report(s).{last}")

    skill = match_skill(task)
    hits = retrieve(task, 8)  # how spread out is this topic across the documents?
    difficulty, why = rule_difficulty(task, hits), 'plain rules: length, scope words, how many documents match'
    text, model = ask('fast', f"Request: {task}\nDocuments that match: {sorted({h['source'] for h in hits})}", ORCH_PROMPT)
    opinion = parse_json(text, {}) if text else {}
    if opinion.get('difficulty') in PLAN:
        difficulty, why = opinion['difficulty'], opinion.get('why', '')
    plan = PLAN[difficulty]
    lineup = (['reader'] if plan['read'] else []) + ['scout', 'writer', 'checker', 'publisher']
    work('orch', f"Skill: {skill['name']}. Difficulty: {difficulty} ({why}). Tier: {plan['tier']}.", model)
    trace.append(handoff('orch', lineup[0], 'Assign the crew', ' → '.join(ICON[a] for a in lineup)))
    return {'request': task, 'skill': skill, 'difficulty': difficulty, 'tier': plan['tier'], 'plan': plan,
            'rules': rules, 'history': history, 'round': 0, 'models': [model], 'trace': trace,
            'notes': [note(state, 'orch', f"{difficulty} task, skill {skill['name']}, tier {plan['tier']}")]}

**4d. Reader.** Finds and opens documents through MCP and writes notes.

In [11]:
# Part 4d · Reader: MCP search + read, then notes.
READER_PROMPT = """You are the Reader. You take faithful notes from local technical documents.
Only write what the documents say. Reply with only JSON: {"notes": [{"file": "<path>", "points": ["<short point>", ...]}]}
with 2 to 4 points per file."""

def plain_points(text):
    """No model: the first sentence under each heading."""
    pts = []
    for block in re.split(r'\n(?=#)', text):
        s = sentences(block)
        if s:
            pts.append(s[0])
    return pts[:4]

async def reader(state: TeamState):
    found = await DOCS_MCP.call('search_files', query=state['request'], limit=state['plan']['read'])
    texts = {f['path']: (await DOCS_MCP.call('read_file', path=f['path']))['text'] for f in found}
    work('reader', f"MCP search_files → {len(texts)} file(s); read_file on {', '.join(texts)}.")
    prompt = '\n\n'.join(f"### {p}\n{t[:2500]}" for p, t in texts.items()) + f"\n\nRequest: {state['request']}"
    text, model = ask('fast', prompt, READER_PROMPT)
    notes = parse_json(text, {}).get('notes') if text else None
    if not isinstance(notes, list) or not notes:
        notes, model = [{'file': p, 'points': plain_points(t)} for p, t in texts.items()], 'plain rules'
    pad = [note(state, 'reader', str(pt), file=n.get('file', '')) for n in notes for pt in n.get('points', [])][:20]
    work('reader', f"{len(pad)} notes on the scratchpad.", model)
    return {'files': list(texts), 'notes': pad, 'models': [model],
            'trace': [handoff('reader', 'scout', 'Notes to verify', f"{len(pad)} notes from {len(texts)} files")]}

**4e. Notes Scout.** Retrieves the evidence passages, numbers them [S1], [S2] and so on, and checks the Reader's notes against them.

In [12]:
# Part 4e · Notes Scout: retrieval + review of the Reader's notes.
SIM = 0.5  # cosine similarity (MiniLM, normalised) a claim needs to count as supported

def best_support(claims, evidence):
    """For each claim, the best-matching evidence sentence or passage: (score, source id)."""
    pool = [(e['sid'], s) for e in evidence for s in sentences(e['text'])] + [(e['sid'], e['text']) for e in evidence]
    sims = embed(claims) @ embed([s for _, s in pool]).T
    return [(float(row.max()), pool[int(row.argmax())][0]) for row in sims]

async def scout(state: TeamState):
    plan = state['plan']
    reader_notes = [n for n in state.get('notes', []) if n['agent'] == 'reader' and n['run'] == state['run_id']]
    queries = [state['request']] + ([n['text'] for n in reader_notes][:3] if state['difficulty'] == 'hard' else [])
    seen = {}
    for q in queries:
        for h in retrieve(q, plan['k']):
            if h['id'] not in seen or h['score'] > seen[h['id']]['score']:
                seen[h['id']] = h
    hits = sorted(seen.values(), key=lambda h: -h['score'])[:plan['k']]
    evidence = [{**h, 'sid': f'S{i + 1}'} for i, h in enumerate(hits)]
    work('scout', f"RAG: {len(queries)} quer{'y' if len(queries) == 1 else 'ies'} → {len(evidence)} passages from "
                  f"{len({e['source'] for e in evidence})} documents.")

    verified, flagged = [], []
    if reader_notes:
        for n, (score, sid) in zip(reader_notes, best_support([n['text'] for n in reader_notes], evidence)):
            (verified if score >= SIM else flagged).append(n['text'])
        work('scout', f"Reviewed the Reader's {len(reader_notes)} notes: {len(verified)} supported, {len(flagged)} not found in the evidence.")
    msg = f"{len(evidence)} passages, S1–S{len(evidence)}" + (f"; dropped {len(flagged)} unsupported Reader note(s)" if flagged else '')
    return {'evidence': evidence, 'verified': verified,
            'notes': [note(state, 'scout', f'unsupported Reader note: {f}') for f in flagged] or [note(state, 'scout', msg)],
            'trace': [handoff('scout', 'writer', 'Evidence pack', msg)]}

**4f. Writer, Checker and Publisher.** The Writer drafts the report from the evidence. The Checker compares every sentence with the evidence and makes sure every file the Reader opened is cited. If anything is wrong, the draft goes back to the Writer. The Publisher removes any claim that is still unsupported, adds the sources and saves the report to long-term memory.

*Limit:* the Checker measures how closely each sentence matches the evidence. It catches claims with no support, but not a single wrong item inside a sentence that otherwise matches (for example, one agent listed under the wrong tier).

In [13]:
# Part 4f · Writer
WRITER_PROMPT = """You are the Technical Writer. Write a clear Markdown report using ONLY the numbered evidence passages.
Cite every factual sentence with its passage id, like [S3], at the end of the same sentence. Never invent facts, numbers
or names, and copy lists, names and numbers exactly as the evidence gives them. If the evidence doesn't cover something, say so. Start with a level-1 title, then use exactly these level-2 sections in this order:
{sections}. Follow the user's standing rules.
Keep it short: about {limit} words in total. Write plain, direct sentences, the way a student writes a technical report.
No filler or hype words (seamless, robust, leverage, delve, crucial, in conclusion), no emoji, no bold inside sentences.
Use bullets only for real lists. Reply with the Markdown only."""

def required_sections(state):
    """The skill's sections, plus any section a standing rule asks for (e.g. "always end with a 'Risks' section")."""
    extra = []
    for r in state.get('rules', []):
        m = re.search(r"[\"'“‘]([^\"'”’]+)[\"'”’]\s+section", r) or re.search(r'\b(?:a|an)\s+([\w &/-]+?)\s+section', r, re.I)
        if m:
            extra.append(m.group(1).strip().capitalize())
    return state['skill']['sections'] + [s for s in extra if s not in state['skill']['sections']]

def plain_section(title, state, used):
    """No model: the evidence sentences closest to the section's topic, cited."""
    topic = state['request'] + ' ' + title
    if re.search(r'risk|limit', title, re.I):
        topic = 'risks limitations caveats fails sleeps resets wiped cannot only off by default may log prompts'
    pool = [(e['sid'], s) for e in state['evidence'] for s in sentences(e['text'])
            if len(s.split()) >= 6 and not re.match(r'(It|They|This|These|That|Those|Its|Their)\b', s)]  # skip sentences that lean on the previous one
    order = (embed([topic]) @ embed([s for _, s in pool]).T)[0].argsort()[::-1]
    picked = [pool[i] for i in order if pool[i][1] not in used][:3 if title in ('Summary', 'Answer') else 4]
    used.update(s for _, s in picked)
    if title in ('Summary', 'Answer'):
        return [' '.join(f"{s} [{sid}]" for sid, s in picked)]
    return [f"- {s} [{sid}]" for sid, s in picked]

def report_title(request):
    """'Write a technical summary report on how X works' -> 'How X Works'."""
    t = re.sub(r'^(please\s+)?(write|give|create|make|prepare|draft)\s+(me\s+)?(an?\s+)?([\w-]+\s+){0,3}?(report|summary|overview|write-?up)\s+(on|of|about|for)\s+', '', request.strip(), flags=re.I)
    t = re.sub(r'[,;]\s*including\b.*$', '', t).rstrip('?.! ')
    small = {'a', 'an', 'and', 'the', 'to', 'of', 'on', 'in', 'for', 'its', 'or', 'with'}
    return ' '.join(w if (k and w.lower() in small) else w[:1].upper() + w[1:] for k, w in enumerate(t.split()))

def plain_draft(state, sections):
    used, out = set(), [f"# {report_title(state['request'])}", '']
    for sec in sections:
        out += [f'## {sec}', *plain_section(sec, state, used), '']
    return '\n'.join(out)

def plain_revise(state, sections):
    """No model: cut what the Checker couldn't ground and add any missing section."""
    draft = state['draft']
    for raw in state['review'].get('unsupported', []):
        draft = draft.replace(raw, '')
    have = {h.strip().lower() for h in re.findall(r'^##\s+(.+)$', draft, re.M)}
    used = set(sentences(draft))
    for sec in sections:
        if sec.lower() not in have:
            draft += '\n\n' + '\n'.join([f'## {sec}', *plain_section(sec, state, used)])
    return re.sub(r'\n(-\s*\n)+', '\n', draft)

async def writer(state: TeamState):
    rnd = state.get('round', 0) + 1
    sections = required_sections(state)
    evidence = '\n\n'.join(f"[{e['sid']}] ({e['source']}) {e['text']}" for e in state['evidence'])
    prompt = (f"Request: {state['request']}\nSkill steps: {state['skill']['steps']}\n"
              f"User's standing rules:\n" + ('\n'.join('- ' + r for r in state.get('rules', [])) or '- none') +
              f"\nReader's verified notes:\n" + ('\n'.join('- ' + n for n in state.get('verified', [])) or '- none') +
              f"\n\nEvidence:\n{evidence}")
    if rnd > 1:
        prompt += f"\n\nYour previous draft:\n{state['draft']}\n\nFix every issue the Checker found:\n" + '\n'.join('- ' + i for i in state['review']['issues'])
    limit = 120 if state['skill']['slug'] == 'quick-answer' else 350
    text, model = ask(state['tier'], prompt, WRITER_PROMPT.format(sections=', '.join(sections), limit=limit))
    if text:
        draft = join_citations(re.sub(r'^```(?:markdown|md)?\s*|\s*```$', '', text.strip()))
        draft = draft.translate({0x2010: '-', 0x2011: '-', 0x202F: ' ', 0x00A0: ' '})  # hyphens/spaces Arial lacks
        for sec in sections:  # some models write sections as level-1 headings
            draft = re.sub(r'(?mi)^#\s+(' + re.escape(sec) + r')\s*$', r'## \1', draft)
    else:
        draft, model = (plain_revise(state, sections) if rnd > 1 else plain_draft(state, sections)), 'plain rules'
    cited = sorted(set(re.findall(r'S\d+', ' '.join(re.findall(r'\[([S\d,\s]+)\]', draft)))), key=lambda s: int(s[1:]))
    work('writer', f"Round {rnd}: {len(draft.split())} words, {len(cited)} sources cited ({', '.join(cited)}).", model)
    return {'draft': draft, 'round': rnd, 'models': [model],
            'notes': [note(state, 'writer', f'draft round {rnd}')],
            'trace': [handoff('writer', 'checker', 'Draft for review', f'round {rnd}')]}

In [14]:
# Part 4f · Checker (fact-check + Reader's coverage review)
def claims(draft):
    """(claim without citations, cited ids, raw sentence) for every factual-looking sentence."""
    out = []
    for s in sentences(draft):
        cites = re.findall(r'S\d+', ' '.join(re.findall(r'\[([S\d,\s]+)\]', s)))
        plain = re.sub(r'\s*\[[S\d,\s]+\]', '', s).strip()
        if len(plain.split()) >= 6:
            out.append((plain, cites, s))
    return out

async def checker(state: TeamState):
    draft, evidence = state['draft'], state['evidence']
    ids = {e['sid'] for e in evidence}
    found = claims(draft)
    issues, unsupported = [], []
    for (plain, cites, raw), (score, sid) in zip(found, best_support([c[0] for c in found], evidence) if found else []):
        if any(c not in ids for c in cites):
            issues.append(f'Cites a source that does not exist: "{plain[:60]}…"')
        if score < SIM:
            unsupported.append(raw)
            issues.append(f'Not supported by the evidence (best match {score:.2f}), remove or rephrase: "{plain[:70]}…"')
        elif not cites:
            issues.append(f'Missing citation, add [{sid}]: "{plain[:60]}…"')
    limit = 120 if state['skill']['slug'] == 'quick-answer' else 350
    words = len(re.sub(r'\[[S\d,\s]+\]', '', draft).split())
    if words > limit * 1.3:  # too long reads like filler; ask for a tighter draft
        issues.append(f'Too long: {words} words. Cut it to about {limit}; keep only what the request needs.')
    have = {h.strip().lower() for h in re.findall(r'^##\s+(.+)$', draft, re.M)}
    issues += [f'Missing section "{s}" (required by the skill or a standing rule).' for s in required_sections(state) if s.lower() not in have]
    # The Reader's peer review: a file it read, that has evidence, should be used.
    cited_sources = {e['source'] for e in evidence if re.search(rf"\b{e['sid']}\b", draft)}
    issues += [f'Reader: {f} was read and has evidence but is never cited.' for f in state.get('files', [])
               if f not in cited_sources and any(e['source'] == f for e in evidence)]

    grounded = len(found) - len(unsupported)
    work('checker', f"{grounded}/{len(found)} claims grounded (similarity ≥ {SIM}); {len(issues)} issue(s).")
    approved = not issues
    last_round = state['round'] >= state['plan']['rounds']
    verdict = 'approved' if approved else ('approved after the last round, unsupported claims will be cut' if last_round else 'sent back')
    review = {'issues': issues, 'unsupported': unsupported, 'approved': approved, 'verdict': verdict, 'grounded': f'{grounded}/{len(found)}'}
    if approved or last_round:
        trace = handoff('checker', 'publisher', 'Approved' if approved else 'Approved with cuts', verdict)
    else:
        trace = handoff('checker', 'writer', 'Send back for revision', ' | '.join(issues[:3]) + (f' (+{len(issues) - 3} more)' if len(issues) > 3 else ''))
    return {'review': review, 'notes': [note(state, 'checker', f"round {state['round']}: {verdict}, {len(issues)} issue(s)")], 'trace': [trace]}

def after_check(state: TeamState):
    """Routing: back to the Writer while there are issues and rounds left, else publish."""
    return 'writer' if not state['review']['approved'] and state['round'] < state['plan']['rounds'] else 'publisher'

In [15]:
# Part 4f · Publisher: combine, enforce grounding, log the action.
def tighten(draft, limit):
    """Over the word limit after the last round: keep up to 4 bullets per section and 3 sentences per paragraph."""
    if len(draft.split()) <= limit * 1.3:
        return draft, False
    out, bullets = [], 0
    for line in draft.splitlines():
        if line.startswith('#'):
            bullets = 0
            out.append(line)
        elif re.match(r'^\s*([-*]|\d+\.)\s+', line):
            bullets += 1
            if bullets <= 4:
                out.append(line)
        elif line.strip():
            parts = re.split(r'(?<=[.!?])\s+(?=[A-Z])', line.strip())
            out.append(' '.join(parts[:3]))
        else:
            out.append(line)
    return '\n'.join(out), True

async def publisher(state: TeamState):
    draft, cut = state['draft'], 0
    for raw in state['review'].get('unsupported', []):  # strict grounding: unsupported claims are never published
        if raw in draft:
            draft, cut = draft.replace(raw, ''), cut + 1
    limit = 120 if state['skill']['slug'] == 'quick-answer' else 350
    draft, trimmed = tighten(draft, limit)
    draft = re.sub(r'\n[-*]\s*(?=\n|$)', '', draft)
    draft = re.sub(r'\n{3,}', '\n\n', draft).strip()
    # A section left empty by the cuts says so, instead of showing an empty heading.
    draft = re.sub(r'^(## .+)\n(?=\s*(?:## |\Z))', r"\1\nThe documents don't cover this.\n", draft + '\n', flags=re.M).strip()
    used = [e for e in state['evidence'] if re.search(rf"\b{e['sid']}\b", draft)]
    report = draft + '\n\n## Sources\n' + '\n'.join(f"- [{e['sid']}] {e['source']}, part {e['chunk'] + 1}" for e in used)
    log_action(state['user_id'], {'at': time.strftime('%Y-%m-%dT%H:%M:%S'), 'request': state['request'], 'skill': state['skill']['name'],
                                  'difficulty': state['difficulty'], 'sources': sorted({e['source'] for e in used}),
                                  'verdict': state['review']['verdict']})
    work('publisher', f"Combined the reviewed draft{f', cut {cut} unsupported claim(s)' if cut else ''}{', trimmed it to the length limit' if trimmed else ''}; {len(used)} sources. Logged to long-term memory.")
    return {'report': report, 'trace': [handoff('publisher', 'you', 'Report ready', f"{len(report.split())} words, {len(used)} sources")]}

## Part 5: Workflow and Memory

- **Short-term memory:** `MemorySaver` saves the team state after every step, for each conversation.
- **Long-term memory:** `InMemoryStore` keeps each user's standing rules and past reports. It is also saved to a JSON file so it survives a restart.

The graph below decides which agent works next.

In [16]:
# Part 5a · Memory.
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore

SHORT_TERM = MemorySaver()
LONG_TERM = InMemoryStore()
MEMORY_FILE = WORK / 'long_term_memory.json'

def _save_memory():
    data = {'/'.join(ns): {i.key: i.value for i in LONG_TERM.search(ns, limit=1000)} for ns in LONG_TERM.list_namespaces()}
    MEMORY_FILE.write_text(json.dumps(data, indent=1), encoding='utf-8')

def _load_memory():
    if MEMORY_FILE.exists():
        for ns, items in json.loads(MEMORY_FILE.read_text(encoding='utf-8')).items():
            for key, value in items.items():
                LONG_TERM.put(tuple(ns.split('/')), key, value)

def remember_rule(uid, rule):
    key = re.sub(r'\W+', '-', rule.lower()).strip('-')[:60]  # same rule twice = one entry
    LONG_TERM.put(('users', uid, 'rules'), key, {'rule': rule, 'added': time.strftime('%Y-%m-%d')})
    _save_memory()

def recall_rules(uid):
    return [i.value['rule'] for i in LONG_TERM.search(('users', uid, 'rules'), limit=50)]

def log_action(uid, action):
    LONG_TERM.put(('users', uid, 'history'), action['at'], action)
    _save_memory()

def recall_history(uid, n=3):
    items = sorted(LONG_TERM.search(('users', uid, 'history'), limit=200), key=lambda i: i.key, reverse=True)
    return [i.value for i in items[:n]]

_load_memory()
print(f"Long-term memory loaded from {rel(MEMORY_FILE)}: {len(LONG_TERM.list_namespaces())} namespace(s).")

Long-term memory loaded from assistant_work/long_term_memory.json: 0 namespace(s).


In [17]:
# Part 5b · Routing: the graph that decides who works next.
from langgraph.graph import StateGraph, START, END

graph = StateGraph(TeamState)
for name, fn in [('orchestrator', orchestrator), ('reader', reader), ('scout', scout),
                 ('writer', writer), ('checker', checker), ('publisher', publisher)]:
    graph.add_node(name, fn)
graph.add_edge(START, 'orchestrator')
graph.add_conditional_edges('orchestrator', lambda s: 'reader' if s['plan']['read'] else 'scout', ['reader', 'scout'])  # easy tasks skip the Reader
graph.add_edge('reader', 'scout')
graph.add_edge('scout', 'writer')
graph.add_edge('writer', 'checker')
graph.add_conditional_edges('checker', after_check, ['writer', 'publisher'])  # the review loop
graph.add_edge('publisher', END)
TEAM = graph.compile(checkpointer=SHORT_TERM, store=LONG_TERM)
print(TEAM.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	orchestrator(orchestrator)
	reader(reader)
	scout(scout)
	writer(writer)
	checker(checker)
	publisher(publisher)
	__end__([<p>__end__</p>]):::last
	__start__ --> orchestrator;
	checker -.-> publisher;
	checker -.-> writer;
	orchestrator -.-> reader;
	orchestrator -.-> scout;
	reader --> scout;
	scout --> writer;
	writer --> checker;
	publisher --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Part 6: Execution

Conversation 1 sets a standing rule and asks for a broad report, so it should be rated hard. Conversation 2 is a new conversation with a short question, so it should be rated easy and skip the Reader. It still follows the rule from conversation 1.

In [18]:
# Part 6 · START
import uuid
from IPython.display import Markdown, display

async def run_team(request, user_id='demo-user', thread_id=None):
    thread_id = thread_id or 'conv-' + uuid.uuid4().hex[:6]
    config = {'configurable': {'thread_id': thread_id}}
    print(f"======== {thread_id} · user {user_id} ========")
    t0 = time.time()
    final = await TEAM.ainvoke({'request': request, 'user_id': user_id, 'run_id': uuid.uuid4().hex[:8]}, config)
    used = sorted(set(final['models']) - {'plain code'})
    print(f"======== done in {time.time() - t0:.0f}s · review: {final['review']['verdict']} "
          f"({final['review']['grounded']} claims grounded) · models: {', '.join(used)} ========")
    return final, config

final1, config1 = await run_team(
    "From now on, always end reports with a 'Risks and limitations' section. "
    "Write a technical summary report on how AssistAnt's Orchestrator routes work to its agents and models, including the fallback chain.",
    thread_id='conversation-1')

======== conversation-1 · user demo-user ========
You → Orchestrator · New request
      From now on, always end reports with a 'Risks and limitations' section. Write a technical summary report on how AssistAnt's Orchestrator routes work to its agents and models, including the fallback chain.
Orchestrator → Memory · Save standing rule
      From now on, always end reports with a 'Risks and limitations' section.
   [Orchestrator · plain code] Long-term memory for demo-user: 1 rule(s), 0 past report(s).


   [Orchestrator · Kilo (keyless)] Skill: Technical report. Difficulty: hard (The request requires synthesizing a comprehensive technical summary report on routing, agents, models, and fallback chains across four different documents.). Tier: best.
Orchestrator → Reader · Assign the crew
      Reader → Notes Scout → Writer → Checker → Publisher
   [Reader · plain code] MCP search_files → 5 file(s); read_file on architecture.md, model-router.md, routines.md, public-demo.md, workspace.md.


   [Reader · Kilo (keyless)] 8 notes on the scratchpad.
Reader → Notes Scout · Notes to verify
      8 notes from 5 files
   [Notes Scout · plain code] RAG: 4 queries → 10 passages from 5 documents.


   [Notes Scout · plain code] Reviewed the Reader's 8 notes: 8 supported, 0 not found in the evidence.
Notes Scout → Writer · Evidence pack
      10 passages, S1–S10


   Router: Kilo (keyless) failed (ReadTimeout), so this went to OVH (keyless).
   [Writer · OVH (keyless)] Round 1: 541 words, 7 sources cited (S1, S2, S3, S4, S6, S7, S8).
Writer → Checker · Draft for review
      round 1


   [Checker · plain code] 21/22 claims grounded (similarity ≥ 0.5); 4 issue(s).
Checker → Writer · Send back for revision
      Not supported by the evidence (best match 0.48), remove or rephrase: "Daily usage caps can force the system to rely on lower-tier or free pr…" | Too long: 541 words. Cut it to about 350; keep only what the request needs. | Reader: public-demo.md was read and has evidence but is never cited. (+1 more)


   [Writer · Kilo (keyless)] Round 2: 543 words, 10 sources cited (S1, S2, S3, S4, S5, S6, S7, S8, S9, S10).
Writer → Checker · Draft for review
      round 2


   [Checker · plain code] 24/24 claims grounded (similarity ≥ 0.5); 1 issue(s).
Checker → Publisher · Approved with cuts
      approved after the last round, unsupported claims will be cut
   [Publisher · plain code] Combined the reviewed draft, trimmed it to the length limit; 6 sources. Logged to long-term memory.
Publisher → You · Report ready
      365 words, 6 sources
======== done in 238s · review: approved after the last round, unsupported claims will be cut (24/24 claims grounded) · models: Kilo (keyless), OVH (keyless) ========


In [19]:
display(Markdown(final1['report']))

# AssistAnt Orchestrator Routing Report

## Summary
The Orchestrator receives every request, matches it against markdown routines with regex triggers, delegates work to tiered sub-agents, selects a model provider from a tier-specific chain, and falls back to rule-based execution when no provider is available [S3][S2][S7][S6].

## Key findings
- Requests are matched against case-insensitive regex triggers; unmatched requests run the unplanned flow and draft a new routine for approval [S2][S3][S4].
- Eleven sub-agents are assigned a tier (fast, smart, best, or code); code-tier agents run instantly without a model [S3][S8].
- The router walks the tier's provider chain and calls the first provider whose health check returns ok; other states are full, off, nokey, or offline [S7].
- Daily caps (e.g., Gemini Flash 20, Groq 1,000) cause providers to be skipped once reached; a 429 marks them exhausted for the day [S6].

## How it works
1. A request arrives at the Orchestrator [S3].
2. The Orchestrator tests all routine triggers and runs the first matching routine; if none match, it executes the unplanned flow and drafts a new routine for approval [S2][S3][S4].
3. The routine's steps are delegated to sub-agents; each agent has a tier that determines its model chain [S3][S8].
4. An agent asks the router for its tier; the router walks the chain—best: Claude Code → Groq gpt-oss-120b → free list → OpenRouter → Ollama; smart: Gemini Flash → Groq gpt-oss-120b → free list → OpenRouter → Ollama → Claude API; fast: Groq gpt-oss-20b → Gemini Flash-Lite → Groq Qwen → free list → OpenRouter → Ollama → Claude API—and selects the first provider with health ok [S7].

## Risks and limitations
- When all providers are down, only rule-based fallback is available, which may reduce output quality [S6].
- Code-tier agents cannot invoke models, limiting them to predefined rule logic [S3][S8].
- Incorrect health-check reporting could cause failed provider calls or unnecessary fallbacks [S7].
- Routine matching uses simple regex; complex intents may fall to the unplanned flow [S2][S3].

## Sources
- [S2] routines.md, part 1
- [S3] architecture.md, part 1
- [S4] routines.md, part 3
- [S6] model-router.md, part 2
- [S7] model-router.md, part 1
- [S8] architecture.md, part 2

In [20]:
# Conversation 2: a new thread, same user. Easy, so no Reader, and the rule from conversation 1 still applies.
final2, config2 = await run_team('What does the knowledge base search use to rank notes?', thread_id='conversation-2')
display(Markdown(final2['report']))

======== conversation-2 · user demo-user ========
You → Orchestrator · New request
      What does the knowledge base search use to rank notes?
   [Orchestrator · plain code] Long-term memory for demo-user: 1 rule(s), 1 past report(s). Last report: “Write a technical summary report on how AssistAnt's Orchestr” on 2026-11-06.


   [Orchestrator · Kilo (keyless)] Skill: Quick answer. Difficulty: easy (The request asks for a specific, single fact regarding the ranking mechanism of the knowledge base search.). Tier: fast.
Orchestrator → Notes Scout · Assign the crew
      Notes Scout → Writer → Checker → Publisher
   [Notes Scout · plain code] RAG: 1 query → 5 passages from 4 documents.
Notes Scout → Writer · Evidence pack
      5 passages, S1–S5


   Router: Kilo (keyless) failed (ReadTimeout), so this went to OVH (keyless).
   [Writer · OVH (keyless)] Round 1: 81 words, 0 sources cited ().
Writer → Checker · Draft for review
      round 1


   [Checker · plain code] 2/4 claims grounded (similarity ≥ 0.5); 4 issue(s).
Checker → Publisher · Approved with cuts
      approved after the last round, unsupported claims will be cut
   [Publisher · plain code] Combined the reviewed draft, cut 2 unsupported claim(s); 1 sources. Logged to long-term memory.
Publisher → You · Report ready
      62 words, 1 sources
======== done in 111s · review: approved after the last round, unsupported claims will be cut (2/4 claims grounded) · models: Kilo (keyless), OVH (keyless) ========


## Answer
The Knowledge Base search ranks notes using a local keyword ranking algorithm based on BM25, with the parameters k1 = 1.4 and b = 0.75.

## Supporting facts
- Search is local keyword ranking with BM25, using k1 = 1.4 and b = 0.75【S1】.

## Risks and limitations
The documents don't cover this.

## Sources
- [S1] knowledge-base.md, part 1

In [21]:
# What the team remembers.
snap = TEAM.get_state(config1)
print(f"Short-term (MemorySaver): {len(list(TEAM.get_state_history(config1)))} checkpoints for conversation-1. Scratchpad:")
for n in snap.values['notes'][:10]:
    print(f"   {ICON[n['agent']]:<16} {n['text'][:95]}")
print('\nLong-term (InMemoryStore), across conversations:')
print('   rules:  ', recall_rules('demo-user'))
for h in recall_history('demo-user'):
    print(f"   history: {h['at']} · {h['skill']} · {h['difficulty']} · “{h['request'][:60]}” · {h['verdict']}")

Short-term (MemorySaver): 10 checkpoints for conversation-1. Scratchpad:
   Orchestrator     hard task, skill Technical report, tier best
   Reader           The Orchestrator receives every request and matches it against routines—markdown files with cas
   Reader           The Orchestrator delegates to eleven subagents, each assigned a tier (fast, smart, best) that d
   Reader           Each request is a run that emits events (plan, agent messages, work done, result) streamed live
   Reader           Each agent requests a tier from the router, which walks that tier's provider chain and calls th
   Reader           The best chain is Claude Code → Groq gpt-oss-120b → free list → OpenRouter → Ollama; smart is G
   Reader           Daily usage caps (e.g., Gemini Flash 20, Groq 1,000) cause providers to be skipped once reached
   Reader           Routines are markdown files with front matter defining a name, flow, and case-insensitive regex
   Reader           Built-in routines include Fini

## Part 7: Final PDF Output

Each report is saved as a PDF in the AUF PTF60 template: the university header, the student table, the report with numbered references, and a one-line note on how it was checked. Set your name, program, professor and date in the first lines of the cell below.

**Where the PDFs are saved**
- On Colab: `/content/assistant/outputs/`. The first PDF also downloads automatically, and both are in the Files panel on the left.
- On your own computer: the `assistant_work/outputs/` folder next to this notebook.

In [22]:
# Part 7 · PDF report in the AUF PTF60 template.
# Fill these in once; DATE_SUBMITTED defaults to the day the notebook runs.
STUDENT_NAME = 'Paul Andrei S. Calma'
PROGRAM = 'BS Computer Science'
PROFESSOR = 'Sir Andrew Ramos'
DATE_SUBMITTED = None  # e.g. 'Nov. 6, 2026'

import datetime, urllib.request
from xml.sax.saxutils import escape
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.units import mm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfgen import canvas as rl_canvas
from reportlab.platypus import Image, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

def _font(name, files, fallback):
    """Use the template's font if this machine has it (Windows, or Liberation Sans on Colab), else a built-in look-alike."""
    for f in files:
        if Path(f).exists():
            try:
                pdfmetrics.registerFont(TTFont(name, f))
                return name
            except Exception:
                pass
    return fallback

WIN, LIB = 'C:/Windows/Fonts/', '/usr/share/fonts/truetype/liberation/'
SANS = _font('Arial', [WIN + 'arial.ttf', LIB + 'LiberationSans-Regular.ttf'], 'Helvetica')
SANS_B = _font('Arial-Bold', [WIN + 'arialbd.ttf', LIB + 'LiberationSans-Bold.ttf'], 'Helvetica-Bold')
SERIF_B = _font('Garamond-Bold', [WIN + 'GARABD.TTF', LIB + 'LiberationSerif-Bold.ttf'], 'Times-Bold')
PAGE = _font('Palatino', [WIN + 'pala.ttf', LIB + 'LiberationSerif-Regular.ttf'], 'Times-Roman')
PAGE_B = _font('Palatino-Bold', [WIN + 'palab.ttf', LIB + 'LiberationSerif-Bold.ttf'], 'Times-Bold')

def seal_path():
    """The AUF seal: next to the notebook (reports/assets), or from the project's GitHub repo."""
    for p in (Path('reports/assets/auf_seal.png'), WORK / 'auf_seal.png'):
        if p.exists():
            return p
    try:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/AkurasKurasa/AssistAnt/main/reports/assets/auf_seal.png', WORK / 'auf_seal.png')
        return WORK / 'auf_seal.png'
    except Exception:
        return None

def today_label():
    d = datetime.date.today()
    month = d.strftime('%B') if len(d.strftime('%B')) <= 4 else d.strftime('%b') + '.'  # "May", "June", "Nov."
    return f'{month} {d.day}, {d.year}'

def small_caps(text, big=24, small=19):
    """ANGELES UNIVERSITY FOUNDATION in the template's style: a larger first letter on each word."""
    return ' '.join(f'<font size="{big}">{w[0]}</font><font size="{small}">{w[1:].upper()}</font>' for w in text.split())

def numbered(story_text, evidence):
    """[S3][S5] -> [1, 2]: number sources by first use, one number per document. Returns text and references."""
    order = []
    story_text = re.sub(r'([.!?])((?:\s*\[[S\d,\s]+\])+)', r'\2\1', story_text)  # 'end. [S1]' -> 'end [S1].'
    def repl(m):
        nums = []
        for sid in re.findall(r'S\d+', m.group(0)):
            src = next((e['source'] for e in evidence if e['sid'] == sid), None)
            if src is None:
                continue
            if src not in order:
                order.append(src)
            n = order.index(src) + 1
            if n not in nums:
                nums.append(n)
        return f" [{', '.join(map(str, sorted(nums)))}]" if nums else ''
    text = re.sub(r'\s*(?:\[[S\d,\s]+\])+', repl, story_text)
    return text, order

class _PageOf(rl_canvas.Canvas):
    """Draws 'Page X of Y' once the total is known."""
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self._pages = []
    def showPage(self):
        self._pages.append(dict(self.__dict__))
        self._startPage()
    def save(self):
        total = len(self._pages)
        for state in self._pages:
            self.__dict__.update(state)
            x, y = 521, 51  # right edge and baseline of the template's page number
            parts = [(PAGE, 'Page '), (PAGE_B, str(self._pageNumber)), (PAGE, ' of '), (PAGE_B, str(total))]
            width = sum(pdfmetrics.stringWidth(t, f, 10) for f, t in parts)
            cx = x - width
            for f, t in parts:
                self.setFont(f, 10); self.drawString(cx, y, t); cx += pdfmetrics.stringWidth(t, f, 10)
            super().showPage()
        super().save()

def build_pdf(final, name):
    path = OUT / f'{name}.pdf'
    body = ParagraphStyle('body', fontName=SANS, fontSize=11, leading=15.5, spaceAfter=7, rightIndent=16)
    h2 = ParagraphStyle('h2', fontName=SANS_B, fontSize=12, leading=15, spaceBefore=9, spaceAfter=4)
    bullet = ParagraphStyle('bullet', parent=body, leftIndent=16, bulletIndent=5, spaceAfter=3, bulletFontName=SANS)
    cell = ParagraphStyle('cell', fontName=SANS, fontSize=11, leading=13)
    note = ParagraphStyle('note', parent=body, fontSize=9.5, leading=13, textColor=colors.HexColor('#444444'))
    center = lambda **k: ParagraphStyle('c', alignment=TA_CENTER, **k)
    page_center = lambda **k: center(rightIndent=36, **k)  # the table runs wider than the text; centre on the page instead

    # Header, as in the template: seal on the left of three centred lines.
    seal = seal_path()
    lines = [Paragraph(small_caps('Angeles University Foundation'), center(fontName=SERIF_B, fontSize=19, leading=19, spaceAfter=10)),
             Paragraph('COLLEGE OF COMPUTER STUDIES', center(fontName=SANS_B, fontSize=14, leading=17)),
             Paragraph('PTF60 – Professional Track 6', center(fontName=SANS, fontSize=14, leading=17))]
    head = Table([[Image(str(seal), width=41.2, height=70.3) if seal else Spacer(1, 1), lines]], colWidths=[87, 370],
                 hAlign='LEFT',
                 style=[('VALIGN', (0, 0), (-1, -1), 'TOP'), ('LEFTPADDING', (0, 0), (0, 0), 36), ('LEFTPADDING', (1, 0), (1, 0), 0), ('RIGHTPADDING', (0, 0), (-1, -1), 0), ('TOPPADDING', (0, 0), (-1, -1), 0)])
    info = Table([[Paragraph(f'Student Name: <font name="{SANS_B}">{escape(STUDENT_NAME)}</font>', cell), Paragraph(f'Date Submitted: {escape(DATE_SUBMITTED or today_label())}', cell)],
                  [Paragraph(f'Program: {escape(PROGRAM)}', cell), Paragraph(f'Professor: {escape(PROFESSOR)}', cell)]],
                 colWidths=[309, 192], rowHeights=[26, 26],
                 style=[('GRID', (0, 0), (-1, -1), 0.6, colors.black), ('VALIGN', (0, 0), (-1, -1), 'TOP'), ('TOPPADDING', (0, 0), (-1, -1), 3)])

    report, refs = numbered(final['report'].split('\n## Sources')[0], final['evidence'])
    lines_md = report.splitlines()
    title = next((l[2:].strip() for l in lines_md if l.startswith('# ')), '')
    if len(title) < 12 or title.lower() in {s.lower() for s in final['skill']['sections']}:  # e.g. a model titled it 'Summary'
        title = final['request']
    title = report_title(title)
    story = [head, Spacer(1, 30), info, Spacer(1, 28),
             Paragraph('Final Project', page_center(fontName=SANS, fontSize=12, leading=15)),
             Paragraph(escape(title), page_center(fontName=SANS_B, fontSize=13, leading=17, spaceBefore=2)), Spacer(1, 6 * mm)]
    plain = lambda s: s.translate({0x2010: '-', 0x2011: '-', 0x202F: ' ', 0x00A0: ' '})  # characters Arial lacks
    text = lambda s: re.sub(r'\*\*(.+?)\*\*', r'\1', escape(plain(s)))  # plain text: no bold inside paragraphs
    seen_title = False
    for line in lines_md:
        l = line.strip()
        if not l:
            continue
        if l.startswith('# ') and not seen_title and l[2:].strip().lower() not in {s.lower() for s in final['skill']['sections']}:
            seen_title = True  # the report title, already shown under "Final Project"
            continue
        if l.startswith('#'):
            story.append(Paragraph(text(l.lstrip('#').strip()), h2))
        elif re.match(r'^([-*]|\d+\.)\s+', l):
            story.append(Paragraph(text(re.sub(r'^([-*]|\d+\.)\s+', '', l)), bullet, bulletText='•'))
        elif not l.startswith('|'):
            story.append(Paragraph(text(l), body))
    if refs:
        story.append(Paragraph('References', h2))
        story += [Paragraph(f'[{i}] {escape(src)}', bullet) for i, src in enumerate(refs, 1)]
    r = final['review']
    story += [Spacer(1, 4 * mm), Paragraph(
        f"Method: written by a team of AI agents from {len(refs)} local document{'s' if len(refs) != 1 else ''}. "
        f"Each statement was checked against the source passages ({r['grounded']} supported) before it was kept.", note)]

    SimpleDocTemplate(str(path), pagesize=A4, title=title, author=STUDENT_NAME, leftMargin=55, rightMargin=39,
                      topMargin=66, bottomMargin=80).build(story, canvasmaker=_PageOf)
    return path

pdfs = [build_pdf(final1, 'assistant_report_conversation1'), build_pdf(final2, 'assistant_report_conversation2')]
for p in pdfs:
    pages = len(re.findall(rb'/Type\s*/Page[^s]', p.read_bytes()))
    print(f"{rel(p)} · {pages} page(s) · {p.stat().st_size / 1024:.0f} KB")
try:
    from google.colab import files
    files.download(str(pdfs[0]))
except ImportError:
    print('Not on Colab: the PDFs are in the folder above.')

assistant_work/outputs/assistant_report_conversation1.pdf · 2 page(s) · 267 KB
assistant_work/outputs/assistant_report_conversation2.pdf · 1 page(s) · 265 KB
Not on Colab: the PDFs are in the folder above.
